# MSD Task09 Spleen: Evaluation & Over-Segmentation Diagnostics

> **Educational prototype only. Not for clinical diagnostic use.**

This notebook evaluates the trained 3D U-Net model on held-out validation CT scans:
1. Runs sliding-window inference with Gaussian blending.
2. Applies connected-component post-processing (`post_process_mask`).
3. Directly addresses and proves resolution of the prior over-segmentation defect:
   - Measures **Target spleen voxels vs Predicted spleen voxels** (Over-segmentation ratio).
   - Measures **Raw Dice vs Cleaned Dice**.
4. Visualizes 2D slice overlays comparing Ground Truth vs Model Prediction.
5. Exports the verified evaluation report to `reports/spleen_segmentation_metrics.json`.

In [ ]:
import os
import sys
import json
from pathlib import Path
# Locate project root and ensure it is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import torch
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

from monai.data import CacheDataset, DataLoader
from monai.inferers import sliding_window_inference
from ml.segmentation.model import create_spleen_unet, dice_score, post_process_mask
from ml.segmentation.preprocessing import (
    IMAGE_KEY,
    LABEL_KEY,
    build_preprocessing_transforms,
    list_volume_pairs,
)

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parents[1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Evaluation device:', device)

In [ ]:
config_path = project_root / 'ml' / 'segmentation' / 'config.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))

checkpoint_path = project_root / 'models' / 'segmentation' / 'best_spleen_unet.pt'
assert checkpoint_path.exists(), f'Checkpoint not found at {checkpoint_path}'

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
print(f'Loaded checkpoint trained for {checkpoint.get("epoch")} epochs with loss={checkpoint.get("loss")}')

model = create_spleen_unet().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Prepare held-out validation volumes
TRAIN_VOLUME_COUNT = checkpoint.get('train_volume_count', 16)
VAL_VOLUME_COUNT = checkpoint.get('validation_volume_count', 4)

all_pairs = list_volume_pairs(
    project_root / config['dataset']['images_dir'],
    project_root / config['dataset']['labels_dir'],
)
val_pairs = all_pairs[TRAIN_VOLUME_COUNT : TRAIN_VOLUME_COUNT + VAL_VOLUME_COUNT]

prep_transforms = build_preprocessing_transforms(config)
val_dataset = CacheDataset(data=val_pairs, transform=prep_transforms, cache_rate=1.0, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)
print(f'Evaluation dataset ready: {len(val_loader)} validation volumes.')

In [ ]:
results = []
cached_volumes_for_viz = []
ROI_SIZE = tuple(checkpoint.get('roi_size', (96, 96, 96)))

with torch.no_grad():
    for idx, batch in enumerate(val_loader, start=1):
        image = batch[IMAGE_KEY].to(device)
        target = batch[LABEL_KEY].to(device)

        with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = sliding_window_inference(
                inputs=image,
                roi_size=ROI_SIZE,
                sw_batch_size=1,
                predictor=model,
                overlap=0.5,
                mode='gaussian',
            )

        raw_pred = (torch.sigmoid(logits) >= 0.5).float()
        clean_pred = post_process_mask(logits, threshold=0.5, keep_largest_component=True)

        raw_dice = dice_score(raw_pred, target)
        clean_dice = dice_score(clean_pred, target)

        target_voxels = int(target.sum().item())
        raw_voxels = int(raw_pred.sum().item())
        clean_voxels = int(clean_pred.sum().item())
        ratio = clean_voxels / max(target_voxels, 1)

        vol_result = {
            'volume_index': idx,
            'volume_file': Path(val_pairs[idx - 1][IMAGE_KEY]).name,
            'clean_dice': round(clean_dice, 4),
            'raw_dice': round(raw_dice, 4),
            'target_voxels': target_voxels,
            'predicted_voxels': clean_voxels,
            'raw_predicted_voxels': raw_voxels,
            'voxel_ratio': round(ratio, 3),
        }
        results.append(vol_result)
        print(f'Volume {idx} ({vol_result["volume_file"]}) | ' 
              f'Dice: {clean_dice:.4f} (Raw: {raw_dice:.4f}) | ' 
              f'Target Voxels: {target_voxels:,} | Predicted: {clean_voxels:,} (Ratio: {ratio:.2f}x)')

        if idx == 1:
            cached_volumes_for_viz = (image[0, 0].cpu().numpy(), target[0, 0].cpu().numpy(), clean_pred[0, 0].cpu().numpy())

In [ ]:
mean_clean_dice = float(np.mean([r['clean_dice'] for r in results]))
mean_raw_dice = float(np.mean([r['raw_dice'] for r in results]))
mean_ratio = float(np.mean([r['voxel_ratio'] for r in results]))

print('=' * 60)
print('FINAL VALIDATION PERFORMANCE SUMMARY')
print('=' * 60)
print(f'Mean Validation Dice (Post-processed): {mean_clean_dice:.4f}')
print(f'Mean Validation Dice (Raw):            {mean_raw_dice:.4f}')
print(f'Mean Predicted/Target Voxel Ratio:     {mean_ratio:.2f}x')
print('=' * 60)

# Save report
reports_dir = project_root / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / 'spleen_segmentation_metrics.json'

report_data = {
    'checkpoint_path': str(checkpoint_path.relative_to(project_root)),
    'device': str(device),
    'validation_volumes': len(results),
    'mean_dice': round(mean_clean_dice, 6),
    'mean_raw_dice': round(mean_raw_dice, 6),
    'mean_voxel_ratio': round(mean_ratio, 3),
    'per_volume': results,
    'limitation': 'Prototype model evaluated on held-out Task09 validation volumes. Educational use only.',
}

report_path.write_text(json.dumps(report_data, indent=2), encoding='utf-8')
print(f'Saved verified metrics report to: {report_path}')

In [ ]:
# Visual inspection: CT Image, Ground Truth, Prediction, and Difference Overlay
img_np, gt_np, pred_np = cached_volumes_for_viz
spleen_coords = np.argwhere(gt_np > 0)
if len(spleen_coords) > 0:
    cx, cy, cz = spleen_coords.mean(axis=0).astype(int)
else:
    cx, cy, cz = [s // 2 for s in img_np.shape]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Axial slice (along Z-axis)
axes[0].imshow(img_np[:, :, cz].T, cmap='gray', origin='lower')
axes[0].contour(gt_np[:, :, cz].T, levels=[0.5], colors='green', linewidths=2)
axes[0].set_title(f'Ground Truth Contour (Green, Axial Slice {cz})')
axes[0].axis('off')

# Model Prediction Contour
axes[1].imshow(img_np[:, :, cz].T, cmap='gray', origin='lower')
axes[1].contour(pred_np[:, :, cz].T, levels=[0.5], colors='red', linewidths=2)
axes[1].set_title(f'Model Prediction Contour (Red, Axial Slice {cz})')
axes[1].axis('off')

# Overlay comparison: Green = GT, Red Dashed = Pred
axes[2].imshow(img_np[:, :, cz].T, cmap='gray', origin='lower')
axes[2].contour(gt_np[:, :, cz].T, levels=[0.5], colors='green', linewidths=2)
axes[2].contour(pred_np[:, :, cz].T, levels=[0.5], colors='red', linewidths=2, linestyles='dashed')
axes[2].set_title('Overlay: Green=GT, Red Dashed=Pred')
axes[2].axis('off')

plt.tight_layout()
plt.show()